# F5 · Vision experiments: CNN vs ViT, robustness, Grad-CAM, ONNX — Feba

Trains ResNet-50 and ViT-B/16 (last 4 blocks unfrozen, 3 epochs, augmentation: crop, flip, JPEG 60–95, blur) on GenImage for the three settings used by the F3 CLIP probe (seen, unseen BigGAN, unseen SD v1.4). It then evaluates all detectors and the CLIP probe clean and under JPEG 50/70 and 20% crop, draws Grad-CAM for 8 images, exports both seen models to ONNX and times them. At the end it pushes the result files to `feba/F5-vision-experiments`.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU** (L4 is faster if available).
2. Colab Secrets (key icon), notebook access on: `GITHUB_TOKEN`, `GIT_EMAIL`, `HF_TOKEN`, `WANDB_KEY`.
3. The GenImage images must be in your Drive at `MyDrive/twincheck/data/processed/vision/genimage/` (they are, from F1).
4. Checkpoints are saved to `MyDrive/twincheck/models/F5/` after every epoch (they are too big for git). If Colab disconnects, run cells 1–2 again and re-run the training cell: `--resume` continues from the last finished epoch.

Expected time on a T4: copy ~10 min, training ~1–1.5 h (6 runs), evaluation ~20 min, Grad-CAM + ONNX ~10 min.

In [ ]:
# 1. Drive, repo on Feba's F5 branch, packages, W&B
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/twincheck'
CKPT = f'{DRIVE}/models/F5'
token = userdata.get('GITHUB_TOKEN')
!git clone -q -b feba/F5-vision-experiments https://{token}@github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!pip install -q -r requirements.txt
!pip install -q onnx
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_KEY')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Copy the GenImage images from Drive to the Colab disk (reading from Drive during training is slow)
ROOT = '/content/genimage_root'
!mkdir -p {ROOT}/data/processed/vision
!cp -r {DRIVE}/data/processed/vision/genimage {ROOT}/data/processed/vision/
!find {ROOT}/data/processed/vision/genimage -type f | wc -l   # should print 18000

In [ ]:
# 3. Quick check on 8 images per generator, 1 epoch (no W&B, not saved to Drive)
!python -m src.vision.train_detector --arch resnet50 --setting seen --root {ROOT} --limit 8 --epochs 1 --batch 8 --ckpt-dir /content/check --out /content/check --no-wandb

In [ ]:
# 4a. ResNet-50: seen, unseen BigGAN, unseen SD (seed 42)
for setting in ['seen', 'unseen_biggan', 'unseen_sd']:
    !python -m src.vision.train_detector --arch resnet50 --setting {setting} --root {ROOT} --ckpt-dir {CKPT} --resume

In [ ]:
# 4b. ViT-B/16: seen, unseen BigGAN, unseen SD (seed 42)
for setting in ['seen', 'unseen_biggan', 'unseen_sd']:
    !python -m src.vision.train_detector --arch vit_b16 --setting {setting} --root {ROOT} --ckpt-dir {CKPT} --resume

In [ ]:
# 4c. OPTIONAL, only if there is time before the freeze: repeat seeds 43 and 44 for the seen setting
# for seed in [43, 44]:
#     for arch in ['resnet50', 'vit_b16']:
#         !python -m src.vision.train_detector --arch {arch} --setting seen --seed {seed} --root {ROOT} --ckpt-dir {CKPT} --resume

In [ ]:
# 5. CNN vs ViT vs CLIP probe, clean and under JPEG 50 / JPEG 70 / 20% crop -> results/F5/cnn_vit.csv
!python -m src.vision.evaluate_detectors --root {ROOT} --ckpt-dir {CKPT}

In [ ]:
# 6. Grad-CAM on 8 validation images -> reports/figures/fig_gradcam_*.png, results/F5/gradcam_*.csv
!python -m src.vision.gradcam --root {ROOT} --ckpt-dir {CKPT}

In [ ]:
# 7. ONNX export + ms per image -> results/F5/onnx_latency.csv (ONNX files stay on Drive)
!python -m src.vision.export_onnx --ckpt-dir {CKPT}

In [ ]:
# 8. Show the result files (copy this output to Claude / the group)
import pandas as pd
for f in ['results/F5/cnn_vit.csv', 'results/F5/onnx_latency.csv']:
    print(f); print(pd.read_csv(f).round(3).to_string(index=False)); print()
!ls results/F5 reports/figures

In [ ]:
# 9. Push the result files (small CSV/JSON/PNG only; checkpoints and ONNX stay on Drive)
git_email = userdata.get('GIT_EMAIL')
!git config user.name "Feba Joshy"
!git config user.email "{git_email}"
!git add results/F5 reports/figures/fig_gradcam_resnet50.png reports/figures/fig_gradcam_vit_b16.png
!git commit -m "[F5] Add CNN vs ViT, robustness, Grad-CAM and ONNX results"
!git push https://{token}@github.com/himaallu/Group-Project.git feba/F5-vision-experiments